# 05 - Binary Classification (Suspicious & Malicious)

This notebook reproduces the binary ML training runs with **resource‑friendly settings** and logs model accuracy/metrics.


## 1) Imports & Paths

In [ ]:
import pandas as pd
import numpy as np
import time

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier

DATA_PATH = "./outputs/ml/ml_train_5m.csv"


## 2) Load Data

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Rows: {len(df):,}")
print(df['label'].value_counts())


## 3) Feature Prep

In [ ]:
feature_cols = [
    "signature_hash_algo",
    "signature_key_algo",
    "public_key_algo",
    "public_key_size",
    "can_issue",
    "pathlen",
    "has_roca",
]
feature_cols = [c for c in feature_cols if c in df.columns]
X = df[feature_cols].copy()

cat_cols = [c for c in ["signature_hash_algo", "signature_key_algo", "public_key_algo", "can_issue", "has_roca"] if c in X.columns]
num_cols = [c for c in ["public_key_size", "pathlen"] if c in X.columns]

preprocess = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            cat_cols,
        ),
        (
            "num",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
            ]),
            num_cols,
        ),
    ]
)


## 4) Binary Model Runner

In [ ]:
def run_binary(label_name: str):
    # Binary target
    y = (df['label'] == label_name).astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    models = {
        "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced"),
        "RandomForest": RandomForestClassifier(n_estimators=150, max_depth=12, n_jobs=-1, class_weight="balanced"),
        "GradientBoosting": GradientBoostingClassifier(n_estimators=100, max_depth=3),
        "HistGradientBoosting": HistGradientBoostingClassifier(max_depth=6),
    }

    results = []
    for name, model in models.items():
        print(f"
Training: {name}")
        start = time.time()
        clf = Pipeline(steps=[("preprocess", preprocess), ("model", model)])
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)

        try:
            if hasattr(clf, "predict_proba"):
                y_scores = clf.predict_proba(X_test)[:, 1]
                roc_auc = roc_auc_score(y_test, y_scores)
            elif hasattr(clf, "decision_function"):
                y_scores = clf.decision_function(X_test)
                roc_auc = roc_auc_score(y_test, y_scores)
            else:
                roc_auc = None
        except Exception:
            roc_auc = None

        elapsed = time.time() - start

        results.append({
            "model": name,
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "roc_auc": roc_auc,
            "train_time_sec": elapsed,
        })

        print(classification_report(y_test, y_pred, digits=4))
        print(f"Time: {elapsed:.2f}s")

    return pd.DataFrame(results)


## 5) Suspicious vs Rest

In [ ]:
susp_results = run_binary("suspicious")
print(susp_results)


## 6) Malicious vs Rest

In [ ]:
mal_results = run_binary("malicious")
print(mal_results)


## 7) Save Results

In [ ]:
susp_results.to_csv('./outputs/ml/model_comparison_suspicious_binary.csv', index=False)
mal_results.to_csv('./outputs/ml/model_comparison_malicious_binary.csv', index=False)
print('Saved CSV results')
